# A：水果分類（白底 + 真實照片）訓練
按順序一格一格執行。每格上面的說明看完再按執行。**先把執行階段改成 GPU**（上方選單：執行階段 → 變更執行階段類型 → T4 GPU）。

> **版本 v3（2026-10-17）**：已刪除「確認 GPU」那一格，步驟編號改成從 1 連續到最後；GPU 請自己在「執行階段 → 變更執行階段類型 → T4 GPU」設定。



## 步驟 1　連接 Google Drive，並準備資料
事先把這兩個檔案放進你的 Google Drive 最上層（我的雲端硬碟）：
- `fruit_AB.zip`（本專案）
- `archive.zip`（Fruits-360 的 Kaggle 下載檔，內含 100x100、original-size、multi 三個資料夾）

**這一格：連接 Google Drive。** 會跳出授權視窗，按允許。之後的檔案都從 Drive 讀、也存回 Drive。

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**步驟 2　解壓縮專案。** 把 Drive 的 `fruit_AB.zip` 複製到 Colab 並解壓縮，再切換到專案資料夾、安裝 `pyyaml`。

In [2]:
# 步驟 2　解壓縮專案
!cp /content/drive/MyDrive/fruit_AB.zip /content/
!unzip -q -o /content/fruit_AB.zip -d /content/
%cd /content/fruit_AB
!pip -q install pyyaml

/content/fruit_AB


**步驟 3　解壓縮 Fruits-360。** 只解出需要的三個資料夾（100x100、original-size、multi），約幾分鐘。最後一行列出資料夾位置，請確認和下一步的路徑一致。

In [3]:
# 步驟 3　解壓縮 Fruits-360（只解需要的三個資料夾，會花幾分鐘）
!unzip -q -o /content/drive/MyDrive/archive.zip "fruits-360_100x100/*" "fruits-360_original-size/*" "fruits-360_multi/*" -d /content/data
# 看一下資料夾位置是不是跟下面的路徑一致
!find /content/data -maxdepth 3 -type d | head -20

/content/data
/content/data/fruits-360_multi
/content/data/fruits-360_multi/test-multiple_fruits
/content/data/fruits-360_100x100
/content/data/fruits-360_100x100/fruits-360
/content/data/fruits-360_100x100/fruits-360/Training
/content/data/fruits-360_100x100/fruits-360/Test
/content/data/fruits-360_original-size
/content/data/fruits-360_original-size/fruits-360-original-size
/content/data/fruits-360_original-size/fruits-360-original-size/Papers
/content/data/fruits-360_original-size/fruits-360-original-size/Training
/content/data/fruits-360_original-size/fruits-360-original-size/Test
/content/data/fruits-360_original-size/fruits-360-original-size/Validation


## 步驟 4　A1：整理真實照片
如果上一格看到的資料夾位置跟下面不一樣，就改 `--src` 的路徑。

**這一格：A1 整理真實照片。** 讀 Fruits-360 的 multi 照片，縮圖、依檔名判斷水果大類並切成 train / val / test；沒有水果（只有蔬菜或堅果）的照片會略過。輸出的表看各水果有幾張。

In [4]:
!python A/A1_prepare_real_photos.py --src /content/data/fruits-360_multi/test-multiple_fruits --out /content/real

找到照片： 919 張
沒有水果（只有蔬菜或堅果）而略過： 180 張

各資料組張數： {'train': 529, 'test': 109, 'val': 101}

各種類（單一水果）或 multi 的 train / val / test 張數：
  apple           188   (126 / 31 / 31)
  cherry           93   (63 / 16 / 14)
  pear             76   (43 / 17 / 16)
  plum             64   (49 / 8 / 7)
  multi            63   (44 / 7 / 12)
  raspberry        51   (37 / 8 / 6)
  peach            39   (26 / 5 / 8)
  nectarine        31   (23 / 1 / 7)
  redcurrant       19   (19 / 0 / 0)
  blackberry       17   (1 / 8 / 8)
  strawberry       16   (16 / 0 / 0)
  banana           11   (11 / 0 / 0)
  lemon            11   (11 / 0 / 0)
  orange            9   (9 / 0 / 0)   <- 張數太少，結果不穩定
  grape             8   (8 / 0 / 0)   <- 張數太少，結果不穩定
  apricot           6   (6 / 0 / 0)   <- 張數太少，結果不穩定
  fig               4   (4 / 0 / 0)   <- 張數太少，結果不穩定
  mango             4   (4 / 0 / 0)   <- 張數太少，結果不穩定
  pomegranate       3   (3 / 0 / 0)   <- 張數太少，結果不穩定
  rambutan          3   (3 / 0 / 0)   <- 張數太少，結果不穩定
  pineapple         

**步驟 4-2　把 A1 的輸出存一份到 Drive。** 之後 Colab_B 挑真實測試照片會用到（`fruit_AB_out/real`）。

In [5]:
# A1 的輸出（縮圖後的真實照片與清單）存一份到 Drive，之後 B 挑測試照片要用
!mkdir -p /content/drive/MyDrive/fruit_AB_out
!cp -r /content/real /content/drive/MyDrive/fruit_AB_out/real

## 步驟 5　A2：建立訓練清單
看輸出表：哪些水果大類沒有真實照片，哪些張數太少。

**這一格：A2 建立訓練清單。** 掃描白底圖與真實照片，列出每個水果大類各有多少張；`沒有真實照片` 的大類只靠白底圖學，結果要保守解讀。

In [6]:
!python A/A2_build_lists.py --fruits100 /content/data/fruits-360_100x100/fruits-360 --original /content/data/fruits-360_original-size/fruits-360-original-size --real_dir /content/real --out /content/lists

掃描 100 Training : 268 個類別
掃描 100 Test : 268 個類別
掃描 original Training : 152 個類別
掃描 original Validation : 152 個類別
掃描 original Test : 152 個類別
水果大類共 55 種
  清單 100_training : 99966 張
  清單 100_test : 33329 張
  清單 original_training : 37911 張
  清單 original_validation : 18960 張
  清單 original_test : 18855 張
真實照片：train 528 / val 101 / test 109 ；因為種類不在 Fruits-360 大類而略過 1 張
  檔名中不在 Fruits-360 水果大類的字（已忽略）： ['kachi', 'tamarilo']

大類   白底100訓練  白底original訓練  真實單一(train/val/test)
apple             16571        11028        126 / 31 / 31
apricot             492            0        6 / 0 / 0
avocado            3060         1428        0 / 0 / 0   <- 沒有真實照片
banana             1917          323        11 / 0 / 0
blackberry         2239         1505        1 / 8 / 8
blueberry           462            0        0 / 0 / 0   <- 沒有真實照片
cactus             1901          940        1 / 0 / 0
cantaloupe         1454          313        0 / 0 / 0   <- 沒有真實照片
carambola          1923          955        0 / 0 / 0   <- 

## 步驟 6　A3：訓練三個模型（每格跑一個模型）
- 每個模型會先用白底圖（100x100 與 original-size 一起用）訓練，再加入真實照片訓練。
- **先把 `--epochs1 1 --epochs2 1` 加上去跑一次看時間**，沒問題再改回預設。
- 想只用 100x100 白底：`--white_source 100`；只用 original-size：`--white_source original`。

**步驟 6-1　訓練 MobileNetV2。** 第一階段只用白底圖訓練最後一層，第二階段加入真實照片、全部參數一起訓練。權重存成 `a_mobilenetv2.pth` 並複製到 Drive。

In [7]:
!python A/A3_train.py --model mobilenetv2 --lists /content/lists --white_source both --epochs1 15 --save_dir /content/fruit_AB/models
!mkdir -p /content/drive/MyDrive/fruit_AB_out
!cp /content/fruit_AB/models/*.pth /content/drive/MyDrive/fruit_AB_out/

使用裝置： cuda
水果大類數： 55
白底訓練圖： 137877 張（來源： both ）
Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth
100% 13.6M/13.6M [00:00<00:00, 121MB/s]
第一階段 epoch 1 / 15  loss = 0.1283  已用 28 秒
第一階段 epoch 2 / 15  loss = 0.056  已用 53 秒
第一階段 epoch 3 / 15  loss = 0.0393  已用 76 秒
第一階段 epoch 4 / 15  loss = 0.0305  已用 99 秒
第一階段 epoch 5 / 15  loss = 0.0252  已用 122 秒
第一階段 epoch 6 / 15  loss = 0.0214  已用 145 秒
第一階段 epoch 7 / 15  loss = 0.0187  已用 166 秒
第一階段 epoch 8 / 15  loss = 0.0166  已用 189 秒
第一階段 epoch 9 / 15  loss = 0.0147  已用 212 秒
第一階段 epoch 10 / 15  loss = 0.0138  已用 234 秒
第一階段 epoch 11 / 15  loss = 0.0128  已用 257 秒
第一階段 epoch 12 / 15  loss = 0.0119  已用 278 秒
第一階段 epoch 13 / 15  loss = 0.011  已用 300 秒
第一階段 epoch 14 / 15  loss = 0.0102  已用 322 秒
第一階段 epoch 15 / 15  loss = 0.0097  已用 343 秒

== 第一階段結束：白底測試 ==
100_test : 張數 5000  第一名正確率 0.87
original_test : 張數 5000  第一名正確率 0.9108

== 第二階段：白底加真實照片 ==  真實訓練 528 張，val 

**步驟 6-2　訓練 ResNet18。** 流程同上，權重存成 `a_resnet18.pth`。

In [8]:
!python A/A3_train.py --model resnet18 --lists /content/lists --white_source both --epochs1 15 --save_dir /content/fruit_AB/models
!cp /content/fruit_AB/models/*.pth /content/drive/MyDrive/fruit_AB_out/

使用裝置： cuda
水果大類數： 55
白底訓練圖： 137877 張（來源： both ）
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100% 44.7M/44.7M [00:00<00:00, 177MB/s]
第一階段 epoch 1 / 15  loss = 0.0967  已用 20 秒
第一階段 epoch 2 / 15  loss = 0.0476  已用 43 秒
第一階段 epoch 3 / 15  loss = 0.0326  已用 64 秒
第一階段 epoch 4 / 15  loss = 0.0251  已用 84 秒
第一階段 epoch 5 / 15  loss = 0.0208  已用 105 秒
第一階段 epoch 6 / 15  loss = 0.0173  已用 127 秒
第一階段 epoch 7 / 15  loss = 0.0154  已用 147 秒
第一階段 epoch 8 / 15  loss = 0.0136  已用 169 秒
第一階段 epoch 9 / 15  loss = 0.0124  已用 190 秒
第一階段 epoch 10 / 15  loss = 0.0116  已用 210 秒
第一階段 epoch 11 / 15  loss = 0.0106  已用 230 秒
第一階段 epoch 12 / 15  loss = 0.0098  已用 250 秒
第一階段 epoch 13 / 15  loss = 0.0093  已用 272 秒
第一階段 epoch 14 / 15  loss = 0.0086  已用 293 秒
第一階段 epoch 15 / 15  loss = 0.0083  已用 314 秒

== 第一階段結束：白底測試 ==
100_test : 張數 5000  第一名正確率 0.8366
original_test : 張數 5000  第一名正確率 0.9042

== 第二階段：白底加真實照片 ==  真實訓練 528 張，val 101 

**步驟 6-3　訓練 Baseline CNN。** 從零開始訓練的對照組，權重存成 `a_baseline.pth`。

In [9]:
!python A/A3_train.py --model baseline --lists /content/lists --white_source both --epochs1 15 --save_dir /content/fruit_AB/models
!cp /content/fruit_AB/models/*.pth /content/drive/MyDrive/fruit_AB_out/

使用裝置： cuda
水果大類數： 55
白底訓練圖： 137877 張（來源： both ）
第一階段 epoch 1 / 15  loss = 0.0942  已用 20 秒
第一階段 epoch 2 / 15  loss = 0.0399  已用 41 秒
第一階段 epoch 3 / 15  loss = 0.0264  已用 61 秒
第一階段 epoch 4 / 15  loss = 0.0202  已用 82 秒
第一階段 epoch 5 / 15  loss = 0.0157  已用 102 秒
第一階段 epoch 6 / 15  loss = 0.0138  已用 122 秒
第一階段 epoch 7 / 15  loss = 0.0116  已用 142 秒
第一階段 epoch 8 / 15  loss = 0.0106  已用 162 秒
第一階段 epoch 9 / 15  loss = 0.0092  已用 182 秒
第一階段 epoch 10 / 15  loss = 0.0096  已用 202 秒
第一階段 epoch 11 / 15  loss = 0.0084  已用 222 秒
第一階段 epoch 12 / 15  loss = 0.0086  已用 243 秒
第一階段 epoch 13 / 15  loss = 0.0083  已用 263 秒
第一階段 epoch 14 / 15  loss = 0.0079  已用 283 秒
第一階段 epoch 15 / 15  loss = 0.0069  已用 304 秒

== 第一階段結束：白底測試 ==
100_test : 張數 5000  第一名正確率 0.8848
original_test : 張數 5000  第一名正確率 0.9108

== 第二階段：白底加真實照片 ==  真實訓練 528 張，val 101 張
第二階段 epoch 1 / 10  loss = 0.2003  真實 val 分數 = 0.1575
第二階段 epoch 2 / 10  loss = 0.1404  真實 val 分數 = 0.1879
第二階段 epoch 3 / 10  loss = 0.1121  真實 val 分數 = 0.2687
第二階段 epoch 4

## 步驟 7　A4：評估
`--stock_dir` 可以放你的 49 張圖庫照片資料夾（檔名像 `Apple_test01.jpg`）。這些照片完全沒參與訓練，是最誠實的測試。不測就拿掉那個參數。

**這一格：A4 評估三個模型。** 白底測試集看第一名正確率；真實單一水果看第一名／前三名；真實多種水果看「找到幾成」。測試集只在這裡用一次。

In [10]:
!python A/A4_evaluate.py --weights /content/fruit_AB/models/a_mobilenetv2.pth --lists /content/lists
!python A/A4_evaluate.py --weights /content/fruit_AB/models/a_resnet18.pth --lists /content/lists
!python A/A4_evaluate.py --weights /content/fruit_AB/models/a_baseline.pth --lists /content/lists

模型： mobilenetv2   大類數： 55
白底 100_test : 第一名 0.9332 （ 5000 張）
白底 original_test : 第一名 0.9498 （ 5000 張）
真實照片 test: {'single_n': 97, 'multi_n': 12, 'single_top1': 0.6494845360824743, 'single_top3': 0.865979381443299, 'multi_found_ratio': 0.5595238095238095, 'multi_any_in_top3': 0.9166666666666666}
每個水果的真實照片正確率已存到 results/a_mobilenetv2_per_family.csv
最常搞混的組合（真實 -> 模型說）：
   blackberry -> raspberry : 5 張
   peach -> apple : 5 張
   pear -> cherry : 3 張
   apple -> cherry : 2 張
   apple -> pear : 2 張
   cherry -> plum : 2 張
   nectarine -> plum : 2 張
   nectarine -> cherry : 2 張
   pear -> apple : 2 張
   blackberry -> strawberry : 1 張
混淆矩陣圖已存到 results/a_mobilenetv2_confusion_real.png
成績已存到 results/a_mobilenetv2_eval.json
模型： resnet18   大類數： 55
白底 100_test : 第一名 0.971 （ 5000 張）
白底 original_test : 第一名 0.985 （ 5000 張）
真實照片 test: {'single_n': 97, 'multi_n': 12, 'single_top1': 0.6597938144329897, 'single_top3': 0.8144329896907216, 'multi_found_ratio': 0.6547619047619048, 'multi_any_in_top3': 0.83333

**步驟 8　把成績與權重存回 Drive。** 最後列出 `fruit_AB_out` 的檔案確認都有存到。把 `a_mobilenetv2.pth`、`a_resnet18.pth`、`a_baseline.pth` 放進網頁程式的 `models/` 資料夾。

In [11]:
# 步驟 8　把成績與權重存回 Drive
!cp /content/fruit_AB/results/* /content/drive/MyDrive/fruit_AB_out/
!ls -la /content/drive/MyDrive/fruit_AB_out/

total 156879
drwx------ 3 root root     4096 Oct 16 14:22 .
drwx------ 5 root root     4096 Oct 16 13:52 ..
-rw------- 1 root root    55514 Oct 16 14:22 a_baseline_confusion_real.png
-rw------- 1 root root    70920 Oct 16 14:22 a_baseline_curves.png
-rw------- 1 root root      383 Oct 16 14:22 a_baseline_eval.json
-rw------- 1 root root      957 Oct 16 14:22 a_baseline_history.json
-rw------- 1 root root     1551 Oct 16 14:22 a_baseline_metrics.json
-rw------- 1 root root      182 Oct 16 14:22 a_baseline_per_family.csv
-rw------- 1 root root 25818613 Oct 16 14:19 a_baseline.pth
-rw------- 1 root root 25818725 Oct 16 14:19 a_baseline_stage1.pth
-rw------- 1 root root    40317 Oct 16 14:22 a_mobilenetv2_confusion_real.png
-rw------- 1 root root    60175 Oct 16 14:22 a_mobilenetv2_curves.png
-rw------- 1 root root      395 Oct 16 14:22 a_mobilenetv2_eval.json
-rw------- 1 root root      954 Oct 16 14:22 a_mobilenetv2_history.json
-rw------- 1 root root     1565 Oct 16 14:22 a_mobilenetv2_